In [ ]:
# 개선된 코드 (undersampling 추가) 2025.12.04 with colpiot

## 📌 `MercariPyCaretAnalyzer` 클래스 요약 문구 

**MercariPyCaretAnalyzer**는 Kaggle *Mercari Price Suggestion Challenge* 데이터를 효율적으로 분석하기 위해 설계된 파이썬 클래스입니다.  
이 클래스는 **데이터 로딩 → 전처리 → 텍스트 벡터화 → PyCaret 환경 설정 → 모델 탐색 및 블렌딩 → 성능 평가 → 시각화 → 제출 파일 생성**까지의 전체 파이프라인을 자동화합니다.  

### 주요 특징
- **데이터 전처리 자동화**: 가격 로그 변환, 카테고리 분해, 희귀 브랜드/카테고리 통합, 텍스트 결측치 처리, 길이 기반 피처 생성.  
- **층화 언더샘플링 지원**: 가격 구간별 균형을 유지하면서 데이터 크기를 줄여 메모리 문제를 해결.  
- **텍스트 벡터화 + 차원 축소**: TF-IDF/Count 기반 벡터화와 TruncatedSVD로 고차원 텍스트 데이터를 효율적으로 처리.  
- **PyCaret 통합**: 손쉽게 회귀 모델을 비교·학습할 수 있도록 PyCaret 환경을 자동 설정.  
- **상위권 모델 블렌딩**: LightGBM, XGBoost, CatBoost, Ridge, Linear Regression 등 Mercari 대회 상위권에서 사용된 모델들을 앙상블.  
- **성능 평가 및 저장**: R², RMSE, MAE를 실제 가격 스케일에서 계산 후 JSON으로 저장.  
- **시각화 및 결과 관리**: 잔차, 피처 중요도 등 시각화를 PNG로 저장하고, 제출 파일을 CSV로 생성하며 모든 결과에 timestamp를 자동 부여.  

### 활용 목적
- Kaggle Mercari Price Suggestion Challenge와 같은 **대규모 가격 예측 문제**에서 빠르고 일관된 분석 파이프라인 제공.  
- **메모리 최적화**와 **상위권 솔루션 전략 반영**으로 실전 대회 환경에 적합.  
- 연구/실무에서 **재현 가능한 실험 관리**를 지원 (결과 파일과 이미지 자동 저장).  

좋습니다 경주님 🙂  
최종 클래스를 Notebook에서 실행하는 **샘플 워크플로우**와, 각 주요 함수의 **처리 흐름 설명문구**를 정리해드릴게요. 이렇게 하면 나중에 클래스 설명서에 그대로 넣을 수 있습니다.

---

## 📓 Jupyter Notebook 샘플 워크플로우

```python
# ============================================================
# 1. 클래스 불러오기 및 객체 생성
# ============================================================
from mercari_pycaret_analyzer import MercariPyCaretAnalyzer

analyzer = MercariPyCaretAnalyzer(
    data_dir="../data",
    images_dir="../images",
    results_dir="../results"
)

# ============================================================
# 2. 데이터 로딩 (층화 언더샘플링 적용)
# ============================================================
analyzer.load_data(train_file="train.tsv", test_file="test.tsv", sep="\t", undersample_frac=0.3)

# ============================================================
# 3. 텍스트 벡터화 + 차원 축소
# ============================================================
analyzer.vectorize_text(
    text_columns=["name", "item_description"],
    method="tfidf",
    max_features=30000,
    n_components=100
)

# ============================================================
# 4. PyCaret Setup
# ============================================================
analyzer.setup_pycaret(session_id=23)

# ============================================================
# 5. 상위권 모델 블렌딩
# ============================================================
best_model = analyzer.find_and_blend_models(sort_metric="R2")

# ============================================================
# 6. 성능 저장 (metrics.json 파일 생성)
# ============================================================
analyzer.save_metrics(model_name="MercariBlended")

# ============================================================
# 7. 시각화 (잔차, 피처 중요도)
# ============================================================
analyzer.visualize_model(plots=["residuals", "feature"])

# ============================================================
# 8. Test 예측 & 제출 파일 생성
# ============================================================
submission = analyzer.predict_test(submission_file="submission.csv")

submission.head()
```

---

## 📌 함수별 처리 흐름 

### `load_data`
- **역할:** 학습/테스트 데이터를 로드하고 기본 전처리를 수행합니다.  
- **처리 흐름:**  
  1. `train.tsv`, `test.tsv` 파일 로드  
  2. `price` 로그 변환 (`np.log1p`)  
  3. 층화 언더샘플링(`_stratified_sample`)으로 가격 구간별 균형 있게 샘플링  
  4. `category_name`을 대/중/소로 분리  
  5. 결측치 처리 및 텍스트 길이 기반 수치 피처 생성  
  6. 희귀 브랜드/카테고리 통합  

---

### `vectorize_text`
- **역할:** 텍스트 데이터를 벡터화하고 차원 축소를 수행합니다.  
- **처리 흐름:**  
  1. `_simple_normalize`로 텍스트 정규화  
  2. `TfidfVectorizer` 또는 `CountVectorizer`로 벡터화  
  3. `TruncatedSVD`로 차원 축소  
  4. 카테고리/길이 피처를 결합하여 최종 학습용 데이터셋 생성  

---

### `setup_pycaret`
- **역할:** PyCaret 환경을 설정합니다.  
- **처리 흐름:**  
  1. 로그 변환된 `price`를 타깃으로 지정  
  2. 카테고리 피처를 지정  
  3. `transformation=False`로 중복 변환 방지  
  4. 데이터 정규화 및 세션 고정  

---

### `find_and_blend_models`
- **역할:** Mercari 상위권 팀들이 사용한 모델들을 학습하고 블렌딩합니다.  
- **처리 흐름:**  
  1. LightGBM, XGBoost, CatBoost, Ridge, Linear Regression 모델 생성  
  2. `blend_models`로 단순 평균 앙상블 수행  
  3. 최적화 기준(`R2`)으로 성능 평가  
  4. 최종 블렌딩 모델을 `best_model`로 저장  

---

### `save_metrics`
- **역할:** 학습 데이터에 대한 성능 지표를 저장합니다.  
- **처리 흐름:**  
  1. `predict_model`로 학습 데이터 예측  
  2. 로그 스케일 예측값을 `expm1`으로 복원  
  3. R², RMSE, MAE 계산  
  4. 결과를 JSON 파일로 저장 (`../results/` 폴더, timestamp 포함)  

---

### `visualize_model`
- **역할:** 모델 성능을 시각화합니다.  
- **처리 흐름:**  
  1. `plot_model`로 잔차, 피처 중요도 등 시각화  
  2. 결과 이미지를 `../images/` 폴더에 저장 (timestamp 포함)  

---

### `predict_test`
- **역할:** 테스트 데이터에 대한 예측을 수행하고 제출 파일을 생성합니다.  
- **처리 흐름:**  
  1. `predict_model`로 테스트 데이터 예측  
  2. 로그 스케일 예측값을 `expm1`으로 복원  
  3. `test_id`와 `price`를 포함한 제출 파일 생성  
  4. 결과를 CSV로 저장 (`../results/` 폴더, timestamp 포함)  

---



In [1]:
import pandas as pd
import numpy as np
import os
import json
import datetime
import gc
from tqdm import tqdm
import warnings
import re

warnings.filterwarnings("ignore")

from pycaret.regression import *
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


class MercariPyCaretAnalyzer:
    def __init__(self,
                 data_dir="../data",
                 images_dir="../images",
                 results_dir="../results"):
        self.data_dir = data_dir
        self.images_dir = images_dir
        self.results_dir = results_dir

        self.train = None
        self.test = None
        self.best_model = None
        self.setup_result = None
        self.metrics = {}

        os.makedirs(self.images_dir, exist_ok=True)
        os.makedirs(self.results_dir, exist_ok=True)

    # 희귀값 통합
    def _collapse_rare_values(self, col, top_k, rare_label="Other"):
        combined = pd.concat([self.train[col], self.test[col]], axis=0)
        value_counts = combined.value_counts()
        top_values = value_counts.index[:top_k]

        self.train[col] = self.train[col].where(self.train[col].isin(top_values), rare_label)
        self.test[col] = self.test[col].where(self.test[col].isin(top_values), rare_label)

    # 텍스트 정규화
    def _simple_normalize(self, text: str) -> str:
        text = str(text).lower()
        text = re.sub(r"[_\-\./]", " ", text)
        text = re.sub(r"\d+", " num ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    # 층화 언더샘플링
    def _stratified_sample(self, frac=0.3, bins=10):
        """
        price 로그 변환된 데이터를 구간별로 나눠 층화 샘플링
        frac: 전체 데이터 중 몇 %를 샘플링할지
        bins: 가격 구간 개수
        """
        # 로그 변환된 price 기준으로 구간 나누기
        self.train["price_bin"] = pd.qcut(self.train["price"], q=bins, duplicates="drop")
        sampled = self.train.groupby("price_bin", group_keys=False).apply(
            lambda x: x.sample(frac=frac, random_state=23)
        )
        self.train = sampled.drop(columns=["price_bin"])
        print(f"⚠️ Stratified undersampling 적용: train {self.train.shape}")

    # 데이터 로딩
    def load_data(self, train_file="train.tsv", test_file="test.tsv", sep="\t", undersample_frac=0.3):
        print("📂 데이터 로딩 시작...")
        train_path = os.path.join(self.data_dir, train_file)
        test_path = os.path.join(self.data_dir, test_file)

        self.train = pd.read_csv(train_path, sep=sep)
        self.test = pd.read_csv(test_path, sep=sep)

        # price 로그 변환
        self.train = self.train[self.train["price"] > 0].dropna(subset=["price"])
        self.train["price"] = np.log1p(self.train["price"])

        # 층화 언더샘플링 적용
        if undersample_frac is not None:
            self._stratified_sample(frac=undersample_frac)

        # category split + 결측치 처리
        for df_name, df in [("train", self.train), ("test", self.test)]:
            df["main_cat"], df["sub_cat"], df["sub_sub_cat"] = zip(
                *df["category_name"].apply(
                    lambda x: (x.split("/") if isinstance(x, str) and "/" in x else ["missing"]*3)
                )
            )
            df["brand_name"] = df["brand_name"].fillna("Unknown").astype(str)
            df["category_name"] = df["category_name"].fillna("Unknown").astype(str)
            df["item_description"] = df["item_description"].fillna("No description").astype(str)
            df["name"] = df["name"].fillna("No name").astype(str)

        # 희귀값 통합
        self._collapse_rare_values("brand_name", top_k=4500, rare_label="Other_brand")
        self._collapse_rare_values("main_cat", top_k=1000, rare_label="Other_main")
        self._collapse_rare_values("sub_cat", top_k=1000, rare_label="Other_sub")
        self._collapse_rare_values("sub_sub_cat", top_k=1000, rare_label="Other_sub_sub")

        # 길이 피처 추가
        for df in [self.train, self.test]:
            df["name_len_char"] = df["name"].str.len()
            df["name_len_word"] = df["name"].str.split().str.len()
            df["desc_len_char"] = df["item_description"].str.len()
            df["desc_len_word"] = df["item_description"].str.split().str.len()

        for df in [self.train, self.test]:
            df["shipping"] = df["shipping"].astype("category")
            df["item_condition_id"] = df["item_condition_id"].astype("category")

        print(f"✅ 데이터 로드 완료: train {self.train.shape}, test {self.test.shape}")

    # 텍스트 벡터화
    def vectorize_text(self, text_columns=["name", "item_description"],
                       method="tfidf", max_features=30000, n_components=100):
        print("📝 텍스트 벡터화 및 차원 축소 시작...")
        for col in text_columns:
            clean_col = f"{col}_clean"
            self.train[clean_col] = self.train[col].apply(self._simple_normalize)
            self.test[clean_col] = self.test[col].apply(self._simple_normalize)

        vectors, feature_names = [], []
        for col in tqdm(text_columns, desc="Text columns"):
            clean_col = f"{col}_clean"
            vec = TfidfVectorizer(max_features=max_features, ngram_range=(1,2)) if method=="tfidf" else CountVectorizer(max_features=max_features, ngram_range=(1,2))
            combined_text = pd.concat([self.train[clean_col], self.test[clean_col]], axis=0)
            vec.fit(combined_text)

            train_vec = vec.transform(self.train[clean_col])
            test_vec = vec.transform(self.test[clean_col])

            if n_components < train_vec.shape[1]:
                svd = TruncatedSVD(n_components=n_components, random_state=23)
                train_vec = svd.fit_transform(train_vec)
                test_vec = svd.transform(test_vec)
            else:
                train_vec = train_vec.toarray()
                test_vec = test_vec.toarray()

            vectors.append((train_vec, test_vec))
            feature_names.append([f"{col}_{i}" for i in range(train_vec.shape[1])])
            gc.collect()

        train_features = np.hstack([v[0] for v in vectors])
        test_features = np.hstack([v[1] for v in vectors])

        self.train_vectorized = pd.DataFrame(train_features, columns=[f for sub in feature_names for f in sub])
        self.test_vectorized = pd.DataFrame(test_features, columns=[f for sub in feature_names for f in sub])

        for col in ["main_cat","sub_cat","sub_sub_cat","brand_name","item_condition_id","shipping",
                    "name_len_char","name_len_word","desc_len_char","desc_len_word"]:
            self.train_vectorized[col] = self.train[col].reset_index(drop=True)
            self.test_vectorized[col] = self.test[col].reset_index(drop=True)

        print(f"✅ 벡터화 완료: train {self.train_vectorized.shape}, test {self.test_vectorized.shape}")

    # PyCaret setup
    def setup_pycaret(self, session_id=23):
        print("🔧 PyCaret setup 시작...")
        categorical_cols = ["main_cat","sub_cat","sub_sub_cat","brand_name","item_condition_id","shipping"]
        existing_categorical = [col for col in categorical_cols if col in self.train_vectorized.columns]

        self.setup_result = setup(
            data=self.train_vectorized.assign(price=self.train["price"].reset_index(drop=True)),
            target="price",
            session_id=session_id,
            categorical_features=existing_categorical if existing_categorical else None,
            normalize=True,
            transformation=False,
            verbose=True,
        )
        print("✅ PyCaret setup 완료")

    # 상위권 모델 블렌딩
    def find_and_blend_models(self, sort_metric="R2"):
        if self.setup_result is None:
            raise ValueError("먼저 setup_pycaret()를 실행하세요.")

        print("🔍 상위권 모델 후보 학습 및 블렌딩 시작...")

        # 후보 모델 생성
        lgbm  = create_model("lightgbm")
        xgb   = create_model("xgboost")
        cat   = create_model("catboost")
        ridge = create_model("ridge")
        lr    = create_model("lr")

        # 블렌딩 (단순 평균 앙상블)
        blended = blend_models([lgbm, xgb, cat, ridge, lr], optimize=sort_metric)

        self.best_model = blended
        print(f"🏆 Blended model 선택 완료 (기준={sort_metric})")
        return self.best_model

    # 성능 저장
    def save_metrics(self, model_name=None):
        if self.best_model is None:
            raise ValueError("모델이 없습니다.")

        pred_df = predict_model(self.best_model, data=self.train_vectorized.copy())
        y_log_true = self.train["price"].values
        y_log_pred = pred_df["Label"].values

        # 로그 스케일 → 실제 가격 스케일 복원
        y_true = np.expm1(y_log_true)
        y_pred = np.expm1(y_log_pred)

        r2 = r2_score(y_true, y_pred)
        rmse = mean_squared_error(y_true, y_pred, squared=False)
        mae = mean_absolute_error(y_true, y_pred)

        self.metrics = {"R2": round(r2,4), "RMSE": round(rmse,4), "MAE": round(mae,4)}

        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        if model_name is None:
            model_name = str(self.best_model).split("(")[0]

        file_path = os.path.join(self.results_dir, f"{model_name}_metrics_{timestamp}.json")
        with open(file_path, "w") as f:
            json.dump(self.metrics, f, indent=4)

        print(f"💾 Metrics 저장 완료: {file_path}")

    # 시각화
    def visualize_model(self, plots=["residuals", "feature"]):
        if self.best_model is None:
            raise ValueError("먼저 find_and_blend_models()로 모델을 선택하세요.")

        print("🎨 시각화 시작...")
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        model_name = str(self.best_model).split("(")[0]

        for p in plots:
            try:
                plot_name = "feature" if p == "feature_importance" else p
                save_path = os.path.join(self.images_dir, f"{model_name}_{plot_name}_{timestamp}.png")
                plot_model(self.best_model, plot=plot_name, save=True)
                print(f"✅ {plot_name} plot 저장 완료: {save_path}")
            except Exception as e:
                print(f"⚠️ Plot {p} 실패: {e}")

    # 테스트 예측 & 제출 파일 생성
    def predict_test(self, submission_file="submission.csv"):
        if self.best_model is None:
            raise ValueError("먼저 find_and_blend_models()로 모델을 선택하세요.")

        print("📦 Test 데이터 예측 시작...")
        predictions = predict_model(self.best_model, data=self.test_vectorized.copy())

        # 로그 스케일 → 실제 가격 복원
        price_log_pred = predictions["Label"].values
        price_pred = np.expm1(price_log_pred)

        submission = pd.DataFrame({"test_id": self.test["test_id"], "price": price_pred})

        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        submission_path = os.path.join(self.results_dir, f"{timestamp}_{submission_file}")
        submission.to_csv(submission_path, index=False)
        print(f"💾 Submission 저장 완료: {submission_path}")
        return submission    
    
# End of class ###############    

In [ ]:
# ============================================================
# 1. 클래스 불러오기 및 객체 생성
# ============================================================
from mercari_pycaret_analyzer import MercariPyCaretAnalyzer

analyzer = MercariPyCaretAnalyzer(
    data_dir="../data",
    images_dir="../images",
    results_dir="../results"
)

# ============================================================
# 2. 데이터 로딩 (층화 언더샘플링 적용)
# ============================================================
analyzer.load_data(train_file="train.tsv", test_file="test.tsv", sep="\t", undersample_frac=0.3)

# ============================================================
# 3. 텍스트 벡터화 + 차원 축소
# ============================================================
analyzer.vectorize_text(
    text_columns=["name", "item_description"],
    method="tfidf",
    max_features=30000,
    n_components=100
)

# ============================================================
# 4. PyCaret Setup
# ============================================================
analyzer.setup_pycaret(session_id=23)

# ============================================================
# 5. 상위권 모델 블렌딩
# ============================================================
best_model = analyzer.find_and_blend_models(sort_metric="R2")

# ============================================================
# 6. 성능 저장 (metrics.json 파일 생성)
# ============================================================
analyzer.save_metrics(model_name="MercariBlended")

# ============================================================
# 7. 시각화 (잔차, 피처 중요도)
# ============================================================
analyzer.visualize_model(plots=["residuals", "feature"])

# ============================================================
# 8. Test 예측 & 제출 파일 생성
# ============================================================
submission = analyzer.predict_test(submission_file="submission.csv")

submission.head()